# Use case 2 (education): the agent as my teaching assistant, not my students'

**STAI-X 2026 SC01 guest segment. Shihao Yang, Georgia Tech ISyE.**

Let me start with what I have **not** figured out.

I do not have a good answer for how students should use AI responsibly in my courses. I have opinions, I have policies in my syllabi, and I am not confident in any of them. If you came to this segment for that, I am going to disappoint you, and I would rather say so than pretend.

What I *have* figured out is the other half of education, the half nobody presents on: **the instructor's back office.** Course sites, rosters, mailing lists, scheduling, reading materials, recordings, transcripts, summaries, and the steady stream of individual student requests. This is most of the hours and none of the glory, and it turns out to be almost perfectly shaped for an agent.

My running example is the **Agentic AI reading group** I started in May 2026. It grew to roughly 28 students across three overlapping email threads. I run it with an agent, and the agent is my TA, not theirs.

## Why the back office is the easy win

The research use case in [notebook 01](01_research_dengue.ipynb) needed constant judgment, because a wrong modeling assumption is invisible and consequential. Administrative work is the opposite:

| | Research modeling | Course administration |
|---|---|---|
| Is a mistake visible? | Often not | Almost always, immediately |
| Cost of a mistake | A wrong scientific claim | A wrong due date, fixed in a minute |
| Does it need domain judgment? | Constantly | Rarely |
| Is it repetitive? | No | Relentlessly |

Everything in the right column says "delegate this." And unlike research code, I do not need to read every line, because **the output is the check**: I look at the course site, and either the dates are right or they are not.

## Job 1: Canvas is an API, and that changes what is worth doing

Every Canvas course site sits on a full REST API (`canvasapi` in Python). Most instructors never touch it, because writing the script costs more than clicking through the UI.

That tradeoff has now flipped, and it flipped in a way that matters more than it sounds.

Here is code I wrote **by hand in August 2024** to roll a semester of assignment due dates forward by a year:

```python
# move_date.py, written by hand, 2024
assignments_due_dates = {
    1809498: '2023-09-04T23:59:00Z',  # HW1
    1809500: '2023-09-18T23:59:00Z',  # HW2
    1809502: '2023-09-25T23:59:00Z',  # HW3
    # ... twelve of these, each ID looked up by hand
}
eastern = pytz.timezone('America/New_York')
for assignment_id, old_due_date in assignments_due_dates.items():
    old_due_date_dt = datetime.strptime(old_due_date, "%Y-%m-%dT%H:%M:%SZ")
    # shift by one year, then re-align to the correct weekday...
```

It works. It also took an evening, most of which was pasting assignment IDs out of the Canvas UI and getting the timezone handling right. **I only ever wrote it because I teach the same course repeatedly.** Anything I do once a year, I clicked through by hand instead.

Today the whole thing is one prompt:

> *Using the `canvasapi` package and my `CANVAS_API_KEY`, list the assignments in course
> 405896. Shift every due date forward by one year, but keep them on the same weekday rather
> than the same calendar date, keep the 23:59 Eastern time, and skip anything that would land
> in the week of fall break. Show me the full before-and-after table and do not write
> anything until I say go.*

Note the last clause. I ask for the table first, every time. That is the entire safety protocol for this kind of work, and it is enough, because a due-date table is something I can check at a glance.

**The real change is not the evening I saved.** It is that the class of tasks worth automating got much larger. Bulk-rewriting rubrics, auditing which assignments lack a description, reconciling the gradebook against a roster, generating twelve self-grading quizzes from one template: I now just do these, because the cost of asking dropped below the cost of clicking.

## Job 2: the mailing list nobody designed

The reading group was not planned. I sent one email to ISyE PhD students in May 2026 asking who was interested. People replied. Then people replied to *forwarded* copies. Then people asked to be added after attending a session. Then a second thread started for a related group, with partial overlap.

Three months later the roster lived in **three email threads and my memory**, which is the normal way this goes and is a genuinely annoying problem: the authoritative membership list exists only as an unstructured conversation.

So I stopped maintaining a list and started reconstructing it on demand:

> *Search my Outlook for every message in the reading group threads since May 1. Extract
> everyone who asked to join, everyone who asked to be removed, and everyone who only ever
> attended. Exclude me. Deduplicate by email, keep the display name from the most recent
> message, and give me a table with which thread each person came in through.*

The agent drives my real mailbox through a small `outlook-cli` skill. Below is that reconciliation on **anonymized** data with the real structure preserved: the same duplicate patterns, the same cross-thread overlap, the same one person who asked to be removed.

In [1]:
import pandas as pd

# Anonymized. Real names and addresses are deliberately not in this public repo.
# The shape is faithful: 3 threads, cross-thread overlap, repeat replies, one removal.
replies = pd.DataFrame([
    # (thread,             person,     address,            date,         intent)
    ("Informal Summer RG", "Student A", "a@example.edu", "2026-05-09", "join"),
    ("Informal Summer RG", "Student B", "b@example.edu", "2026-05-09", "join"),
    ("Informal Summer RG", "Student C", "c@example.edu", "2026-05-09", "join"),
    ("Informal Summer RG", "Student D", "d@example.edu", "2026-05-09", "join"),
    ("Informal Summer RG", "Student E", "e@example.edu", "2026-05-12", "join"),
    ("Informal Summer RG", "Student B", "b@example.edu", "2026-05-12", "join"),   # replied twice
    ("Informal Summer RG", "Shihao Yang", "shihao.yang@isye.gatech.edu", "2026-05-13", "organizer"),
    ("Informal Summer RG", "Student F", "f@example.edu", "2026-05-14", "attended-then-join"),
    ("Informal Summer RG", "Student G", "g@example.edu", "2026-05-14", "cannot-attend"),
    ("Agentic AI RG",      "Student C", "c@example.edu", "2026-05-19", "join"),   # in both threads
    ("Agentic AI RG",      "Student H", "h@example.edu", "2026-05-19", "join"),
    ("Agentic AI RG",      "Student I", "i@example.edu", "2026-05-26", "join"),
    ("Agentic AI RG",      "Student E", "e@example.edu", "2026-05-27", "remove"),  # opted out
    ("Skills RG",          "Student J", "j@example.edu", "2026-06-07", "join"),
    ("Skills RG",          "Student A", "a@example.edu", "2026-06-07", "join"),   # in both threads
])
replies.columns = ["thread", "person", "address", "date", "intent"]
replies["date"] = pd.to_datetime(replies["date"])
print(f"{len(replies)} raw replies across {replies.thread.nunique()} threads")

15 raw replies across 3 threads


In [2]:
JOINED  = {"join", "attended-then-join"}
EXCLUDE = {"organizer"}

r = replies[~replies.intent.isin(EXCLUDE)].sort_values("date")
removed = set(r.loc[r.intent == "remove", "address"])

roster = (r[r.intent.isin(JOINED) & ~r.address.isin(removed)]
          .groupby("address")
          .agg(name=("person", "last"),                          # most recent display name
               first_contact=("date", "min"),
               threads=("thread", lambda s: ", ".join(sorted(set(s)))))
          .sort_values("first_contact")
          .reset_index()[["name", "address", "first_contact", "threads"]])
roster["first_contact"] = roster.first_contact.dt.strftime("%Y-%m-%d")

print(f"Raw replies:        {len(replies)}")
print(f"Excluded (me):      {(replies.intent == 'organizer').sum()}")
print(f"Never joined:       {(replies.intent == 'cannot-attend').sum()}")
print(f"Opted out:          {len(removed)}")
print(f"Final roster:       {len(roster)} people, "
      f"{(roster.threads.str.contains(',')).sum()} in more than one thread\n")
display(roster.style.hide(axis='index'))

Raw replies:        15
Excluded (me):      1
Never joined:       1
Opted out:          1
Final roster:       8 people, 2 in more than one thread



name,address,first_contact,threads
Student A,a@example.edu,2026-05-09,"Informal Summer RG, Skills RG"
Student B,b@example.edu,2026-05-09,Informal Summer RG
Student C,c@example.edu,2026-05-09,"Agentic AI RG, Informal Summer RG"
Student D,d@example.edu,2026-05-09,Informal Summer RG
Student F,f@example.edu,2026-05-14,Informal Summer RG
Student H,h@example.edu,2026-05-19,Agentic AI RG
Student I,i@example.edu,2026-05-26,Agentic AI RG
Student J,j@example.edu,2026-06-07,Skills RG


Twelve lines of pandas, and the mailing list is now a **derived quantity** rather than a thing I maintain. When somebody replies tomorrow, I re-derive it.

Two details in that code are worth pointing at, because they are exactly the details an agent gets right and a hurried human gets wrong:

- **Exclude yourself.** My own messages are in every thread. Forget this and you email yourself forever.
- **Removals must beat joins regardless of order.** Student E asked to join on May 12 and to be removed on May 27. Group-then-filter handles that; filter-then-group would have silently re-added them. Emailing somebody who asked to be left alone is the one error in this whole notebook with an actual human cost.

## Job 3: turning a recording into something people will read

Each session produces an hour of video that, realistically, nobody rewatches. The pipeline that fixes this is four steps, and the agent runs all four:

```
recording ──▶ transcript ──▶ summary + decisions ──▶ email to the roster
   Zoom        MacWhisper        the agent            drafted, I send
```

The prompt is boring, which is the point:

> *Find yesterday's reading group recording, pull the transcript, and write a summary for the
> mailing list: what paper we covered, the three main threads of discussion, the open
> questions we did not resolve, and who volunteered to present next. Keep it under 200 words,
> plain prose, no bullet-point salad. Then draft it to the roster. Do not send.*

Two things I learned making this reliable:

**Audit the transcript before trusting it.** My local transcription tool has a failure mode where the language detector flips an English meeting to Chinese and returns confident, fluent, entirely hallucinated text. A summary of a hallucinated transcript is indistinguishable from a real one. So the transcript step now *checks itself* before the summary step runs. Any pipeline where step N cannot tell that step N-1 failed will eventually publish nonsense.

**"Do not send" is not optional.** Which brings me to the part I care most about.

## The line I actually enforce

I let the agent do anything it wants that I can undo. I do not let it do things that land in another person's inbox or another person's grade.

| Agent does it, no review | I review, then it acts | I do it myself |
|---|---|---|
| Search my mail, build the roster | Send any email to students | Anything touching a grade |
| Pull transcripts, draft summaries | Publish to the Canvas site | Judgment calls on accommodations |
| Read the gradebook, flag anomalies | Bulk due-date changes | Anything I would not sign |
| Draft replies to routine requests | Calendar invites to real people | Recommendation letters |

The rule underneath the table: **reads are free, writes are reviewed, and anything a student experiences as coming from me has to actually have come from me.**

There is also a boundary I will name plainly, because we are a room of statisticians and somebody is going to ask. Student records are FERPA-protected. Rosters, grades, and accommodation requests are not material I hand to an arbitrary third-party service without knowing where it goes. In practice that means the read-heavy work runs against my own mailbox and my own institution's API with my own credentials, and the roster in *this public notebook* is fabricated. That is not paranoia, it is the same care you would take with any identifiable data.

## The thing that made this stick: write the knowledge down once

The first month, I re-explained my setup in every conversation. How to reach my mailbox. That subject lines must be ASCII. That timestamps come back in the wrong timezone. That drafts must never auto-send.

The fix was to stop re-explaining and put it in a file the agent loads automatically. Every agent has some version of this and they are all just markdown, so nothing here is locked to one vendor: Codex reads `AGENTS.md`, Claude Code reads `CLAUDE.md`, and Claude Code additionally lets you package a reusable one as a *skill*. Mine for email is about a page of prose plus a companion file of accumulated gotchas, and it turned "drive my Outlook" from a fragile negotiation into one line.

The gotchas file is the part I underrate least. Every time something breaks in a confusing way, that lesson gets appended:

```markdown
### SSH timeout is not a broker failure
If the connection itself hangs, the Windows box is asleep. Tailscale will report the
node active for a short window via cached heartbeat during sleep, which is misleading.
Ask the user to wake it; do not try to restart the broker.

### ASCII subjects only
Em-dashes and smart quotes get mangled in the SSH to Windows to COM pipeline.
Bodies delivered via stdin are safe for any Unicode.
```

Neither of those is knowledge an agent can derive. Both cost me a confusing afternoon exactly once. This file is the difference between a demo and something I actually rely on, and if you take one operational habit from this segment, take this one: **when you debug something surprising, write it down where the agent will read it next time.**

## More prompts in the same family

Everything above is one prompt each. These are the others I reach for often enough to have memorized the shape, kept here so this notebook is the only place you need to look.

**Canvas, read-only audits.** Safe to run without thinking, because they change nothing:

> *Audit course 405888: list every assignment with no description, no due date, or zero points,
> and every quiz that is unpublished. Just the table, no changes.*

> *Compare the Canvas gradebook against `roster.csv`. Who is enrolled but has no submissions,
> and who has submissions but is not on the roster?*

**Canvas, bulk creation.** Note the "list what you are about to create first":

> *Duplicate the quiz "HW1 self-grading" in course 405888 six times, named HW2 through HW7,
> keeping every setting identical, all unpublished. List what you are about to create first.*

**The one to run after any of them:**

> *Walk me through what you just did. Where did you have to make a choice I did not specify?*

That is the same question as in [notebook 01](01_research_dengue.ipynb), and it earns its keep just as often here. The difference is that a wrong due date announces itself the moment you look at the course site, whereas a wrong modeling assumption does not. Administrative work is delegable precisely because its mistakes are loud.

## Honest scorecard

**Works well, use it tomorrow.** Reconstructing structure from unstructured email. Bulk operations against Canvas. Transcript to summary. Drafting the fifteenth reply to the same routine question. None of this needs a clever agent; the basic capability is already enough.

**Works, with a human gate.** Anything outbound. Not because the drafts are bad, they are usually better than my own tired 11pm phrasing, but because I want the responsibility to stay attached to a person.

**Does not work yet.** The thing I opened with. I do not have a defensible answer for student use of AI in assessment, and I notice that I have quietly gotten very good at using these tools for my own work while still not knowing what to tell a first-year student who asks whether they may use one on a homework. That gap is uncomfortable and I think it is the more important problem. I would genuinely like to hear from this room in the Q&A.

---

**Back to:** [`00_start_here.md`](../00_start_here.md) for the summary and the takeaways.